In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv


In [2]:
temp_df = pd.read_csv('/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv')

In [3]:
df = temp_df.iloc[:10000]
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.drop_duplicates(inplace=True)

/tmp/ipykernel_37/3006716147.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.drop_duplicates(inplace=True)


In [5]:
#remove HTML tags
import re
def remove_tags(raw_text):
    cleaned_text = re.sub(re.compile('<.*?>'), '', raw_text)
    return cleaned_text

In [6]:
df['review'] = df['review'].apply(remove_tags)

/tmp/ipykernel_37/2336150696.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(remove_tags)


In [7]:
df['review'] = df['review'].apply(lambda x:x.lower())

/tmp/ipykernel_37/740760900.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x:x.lower())


In [8]:
from nltk.corpus import stopwords

sw_list = stopwords.words('english')

df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))

/tmp/ipykernel_37/2826946130.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['review'] = df['review'].apply(lambda x: [item for item in x.split() if item not in sw_list]).apply(lambda x:" ".join(x))


In [9]:
import gensim

In [10]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

In [11]:
story = []
for doc in df['review']:
    raw_sent = sent_tokenize(doc)
    for sent in raw_sent:
        story.append(simple_preprocess(sent))

In [12]:
model = gensim.models.Word2Vec(
    window=10,
    min_count=2
)

In [13]:
model.build_vocab(story)

In [17]:
model.train(story, total_examples=model.corpus_count, epochs=5)

(5850537, 6186875)

In [19]:
#the output would be these many words it has created vectors
len(model.wv.index_to_key)

31845

In [20]:
# it creates the reviews into vectors
def document_vector(doc):
    # remove out-of-vocabulary words
    doc = [word for word in doc.split() if word in model.wv.index_to_key]
    return np.mean(model.wv[doc], axis=0)

In [21]:
document_vector(df['review'].values[0])

array([ 0.20207678, -0.19886379, -0.11018904, -0.25730416,  0.10482732,
       -0.1589644 ,  0.27808058,  0.6414785 , -0.00668931, -0.0851499 ,
       -0.06131419, -0.35426217, -0.06687965,  0.40400773,  0.38775548,
       -0.11341362, -0.04159196, -0.4625547 , -0.01640911, -0.19756724,
        0.3280529 , -0.15122408, -0.11080384, -0.07045402, -0.2764632 ,
       -0.26342958,  0.08858179, -0.3465829 , -0.2480807 , -0.04617323,
        0.1830098 , -0.08632372,  0.10373057, -0.25272897, -0.39028627,
        0.02086978, -0.0808911 , -0.31979275, -0.12385984, -0.662657  ,
        0.3149719 , -0.02169261, -0.20597026, -0.07518419,  0.2461376 ,
       -0.07363299,  0.09821369, -0.3179174 ,  0.44003546,  0.18568091,
       -0.03918074, -0.14599729, -0.00936463, -0.04574617, -0.3201159 ,
        0.25467962,  0.10894845,  0.35290822, -0.14481398,  0.33579698,
       -0.28857905,  0.25478223, -0.3389357 ,  0.12054663,  0.0102171 ,
        0.4120928 ,  0.1882601 ,  0.16157553, -0.23208591,  0.13

In [22]:
from tqdm import tqdm

In [24]:
#we will convert each review into numbers and taking mean
X = []
for doc in tqdm(df['review'].values):
    X.append(document_vector(doc))

100%|██████████| 9983/9983 [05:55<00:00, 28.09it/s]


In [25]:
X = np.array(X)

In [26]:
X[0]

array([ 0.20207678, -0.19886379, -0.11018904, -0.25730416,  0.10482732,
       -0.1589644 ,  0.27808058,  0.6414785 , -0.00668931, -0.0851499 ,
       -0.06131419, -0.35426217, -0.06687965,  0.40400773,  0.38775548,
       -0.11341362, -0.04159196, -0.4625547 , -0.01640911, -0.19756724,
        0.3280529 , -0.15122408, -0.11080384, -0.07045402, -0.2764632 ,
       -0.26342958,  0.08858179, -0.3465829 , -0.2480807 , -0.04617323,
        0.1830098 , -0.08632372,  0.10373057, -0.25272897, -0.39028627,
        0.02086978, -0.0808911 , -0.31979275, -0.12385984, -0.662657  ,
        0.3149719 , -0.02169261, -0.20597026, -0.07518419,  0.2461376 ,
       -0.07363299,  0.09821369, -0.3179174 ,  0.44003546,  0.18568091,
       -0.03918074, -0.14599729, -0.00936463, -0.04574617, -0.3201159 ,
        0.25467962,  0.10894845,  0.35290822, -0.14481398,  0.33579698,
       -0.28857905,  0.25478223, -0.3389357 ,  0.12054663,  0.0102171 ,
        0.4120928 ,  0.1882601 ,  0.16157553, -0.23208591,  0.13

In [27]:
#we will change sentiment +ve and -ve won't be understanable, so we convert 1 and 0
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y = encoder.fit_transform(df['sentiment'])

In [28]:
y

array([1, 1, 1, ..., 0, 0, 1])

In [29]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test  = train_test_split(X,y,test_size=0.2,random_state=1)

In [31]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [32]:
rf = RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred = rf.predict(X_test)
accuracy_score(y_test,y_pred)

0.8147220831246871